## Import Libraries

In [ ]:
import numpy as np
import pandas as pd

from PIL import Image as PImage, ImageDraw as PImageDraw

from datasets import load_dataset

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

### Silhouette `Dataset`
- [visualizedata/revolutionary_silhouettes](https://huggingface.co/datasets/visualizedata/revolutionary_silhouettes)

In [ ]:
sil_ds = load_dataset("visualizedata/revolutionary_silhouettes", split="silhouetted", streaming=True)
sil_ds

In [ ]:
xy_feats = [f for f in sil_ds.features.keys() if f.startswith(("x", "y"))]
non_xy_feats = [f for f in sil_ds.features.keys() if not f.startswith(("x", "y"))]

sil_xy_ds = sil_ds.select_columns(xy_feats)

sil_xy_df = sil_xy_ds.to_pandas()
sil_xy_df

### Clustering of Silhouettes

In [ ]:
# Center points
xy_pairs = sil_xy_df.values.reshape(len(sil_xy_df), -1, 2)
xy_mean = xy_pairs.mean(axis=1).mean(axis=0)
xy_centered = (xy_pairs - xy_mean).reshape(len(sil_xy_df), -1)
sil_xy_cetered_df = pd.DataFrame(xy_centered, columns=sil_xy_df.columns)

In [ ]:
n_clusters = 8

kmeans = KMeans(n_clusters=n_clusters, random_state=1010).fit(sil_xy_cetered_df)

In [ ]:
r = 10

for cidx in range(n_clusters):
  aimg = PImage.fromarray(255*np.ones((2000,2000,3), dtype=np.uint8)).convert("RGBA")
  aiw,aih = aimg.size
  draw = PImageDraw.Draw(aimg)

  cluster_sils = sil_xy_df.iloc[kmeans.labels_ == cidx]
  cluster_avg = cluster_sils.values.mean(axis=0).reshape(-1,2)
  dists = euclidean_distances([kmeans.cluster_centers_[cidx]], cluster_sils)
  by_dist = dists[0].argsort()

  for x,y in cluster_sils.values[by_dist[0]].reshape(-1,2):
    draw.ellipse((x-r, y-r, x+r, y+r), fill=(0,0,0))

  for x,y in kmeans.cluster_centers_[cidx].reshape(-1,2):
    x += xy_mean[0]
    y += xy_mean[1]
    draw.ellipse((x-r, y-r, x+r, y+r), fill=(220,0,0,100))
  display(aimg.resize((200, 200)))